# SC4002: Natural Language Processing Project

## Task: Build a system that identifies user's intent and extracts the slots in an English Assistance request. It involves two stages:
1. Task 1: Develop a baseline independently, without Generative AI assistance.
2. Task 2: Collaborate with AI to diagnose weaknesses, implement improvements.

Evaluate the resulting models from the above two stages. Submit reproducible notebook evidence, chat history with AI and a report of no more than two pages.

* Data can be found here "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz", en-US.jsonl file

In [77]:
# Necessary Files needed for Project
import re
import time
import json
import numpy as np
import pandas as pd
import sklearn
import random
import joblib
import torch

from pathlib import Path
# Dataset path relative to the notebook's working directory
DATA_PATH = (
    Path("amazon-massive-dataset-1.0")
    / "1.0"
    / "data"
    / "en-US.jsonl"
)

# Visualisation Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, log_loss
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import SGDClassifier
from collections import Counter
from scipy.sparse import hstack, vstack, csr_matrix

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [78]:
# Verify that the dataset exists
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH.resolve()}"

# Load MASSIVE en-US dataset
try: 
    df = pd.read_json(DATA_PATH, lines=True)
    print(df.head())
    print(df.columns)
except FileNotFoundError:
    print("ERROR: File not found")
except json.JSONDecodeError:
    print("ERROR: Invalid Json format in the file")

   id locale partition scenario             intent  \
0   0  en-US      test    alarm          alarm_set   
1   1  en-US     train    alarm          alarm_set   
2   2  en-US     train    alarm          alarm_set   
3   3  en-US      test    audio  audio_volume_mute   
4   4  en-US     train    audio  audio_volume_mute   

                                   utt  \
0      wake me up at five am this week   
1      wake me up at nine am on friday   
2  set an alarm for two hours from now   
3                                quiet   
4                           olly quiet   

                                           annot_utt  worker_id  
0  wake me up at [time : five am] [date : this week]          1  
1  wake me up at [time : nine am] on [date : friday]          1  
2       set an alarm for [time : two hours from now]          1  
3                                              quiet          1  
4                                         olly quiet          1  
Index(['id', 'locale', 'pa

In [79]:
# Create official splits
train_df = df[df['partition'] == 'train'].copy()
dev_df = df[df['partition'] == 'dev'].copy()
test_df = df[df['partition'] == 'test'].copy()

# Print split sizes
print("Train:", len(train_df))
print("Dev:", len(dev_df))
print("Test:", len(test_df))

# Verify and assert split sizes
assert len(train_df) == 11514
assert len(dev_df) == 2033
assert len(test_df) == 2974

print("Dataset split verification passed.")

Train: 11514
Dev: 2033
Test: 2974
Dataset split verification passed.


In [80]:
# First, find out what are the different values in 'intent'
print(df['intent'].unique())

['alarm_set' 'audio_volume_mute' 'iot_hue_lightchange' 'iot_hue_lightoff'
 'iot_hue_lighton' 'iot_hue_lightdim' 'iot_cleaning' 'calendar_query'
 'play_music' 'general_quirky' 'general_greet' 'datetime_query'
 'datetime_convert' 'takeaway_query' 'alarm_remove' 'alarm_query'
 'news_query' 'music_likeness' 'music_query' 'iot_hue_lightup'
 'takeaway_order' 'weather_query' 'music_settings' 'audio_volume_down'
 'general_joke' 'music_dislikeness' 'audio_volume_other' 'iot_coffee'
 'audio_volume_up' 'iot_wemo_on' 'iot_wemo_off' 'qa_stock' 'play_radio'
 'social_post' 'recommendation_locations' 'cooking_recipe' 'qa_factoid'
 'recommendation_events' 'calendar_set' 'play_audiobook' 'play_podcasts'
 'social_query' 'transport_query' 'email_sendemail' 'transport_ticket'
 'recommendation_movies' 'lists_query' 'play_game' 'email_query'
 'transport_traffic' 'cooking_query' 'qa_definition' 'calendar_remove'
 'lists_remove' 'email_querycontact' 'lists_createoradd'
 'email_addcontact' 'transport_taxi' 'qa_

In [81]:
# Parse Slots Function
def parse_slots(annot_utt): 
    """
    Converts 'wake me up at [time:five am]'
    into tokens: ['wake', 'me', 'up', 'at', 'five', 'am']
    and tags: ['0','0','0','0','B-time', 'I-time']
    """

    if not isinstance(annot_utt, str):
        return [], []
    
    tokens = []
    tags = []

    # This regex finds the [type:value] patterns
    pattern = r'\[([^\]:]+)\s*:\s*([^\]]+)\]'
    last_end = 0

    for match in re.finditer (pattern, annot_utt):
        # Handles 'O' tags before the Slot
        before_text = annot_utt[last_end:match.start()].strip()
        if before_text:
            words = before_text.split()
            tokens.extend(words)
            tags.extend(['O'] * len (words))

        # Handles the Slot
        slot_type = match.group(1).strip()
        slot_value = match.group(2).strip()
        slot_words = slot_value.split()

        if slot_words:
            tokens.extend(slot_words)
            tags.append(f'B-{slot_type}') # First word is Beginning
            tags.extend([f'I-{slot_type}'] * (len(slot_words) - 1)) # Rest are Inside

        last_end = match.end()
        
    # Handles text after final slot
    after_text = annot_utt[last_end:].strip()
    if after_text:
        words = after_text.split()
        tokens.extend(words)
        tags.extend(['O'] * len(words))

    return tokens, tags

# Apply cleaning to the utterance ('utt' column, Task 1)
#train_df['utt_cleaned'] = train_df['utt'].apply(clean_text)
#test_df['utt_cleaned'] = test_df['utt'].apply(clean_text)

# Encode the Intent Labels
label_encoder = LabelEncoder()

train_labels = label_encoder.fit_transform(train_df['intent']) #learn from training data only
dev_labels = label_encoder.transform(dev_df['intent'])
test_labels = label_encoder.transform(test_df['intent'])

assert len(label_encoder.classes_) == 60

# Parse slot annotations for all 3 splits
train_df[['tokens', 'slot_tags']] = (train_df['annot_utt'].apply(lambda x: pd.Series(parse_slots(x))))

dev_df[['tokens', 'slot_tags']] = (dev_df['annot_utt'].apply(lambda x: pd.Series(parse_slots(x))))

test_df[['tokens', 'slot_tags']] = (test_df['annot_utt'].apply(lambda x: pd.Series(parse_slots(x))))

# Check tokens and tags match
for name, split_df in[('Train', train_df), ('Dev', dev_df), ('Test', test_df)]:
    
    assert all(
        len(tokens) == len(tags)
        for tokens, tags in zip(split_df['tokens'], split_df['slot_tags'])
    )

    print(f"{name}: token and tag lengths match")
    

Train: token and tag lengths match
Dev: token and tag lengths match
Test: token and tag lengths match


In [82]:
# Inspect a few examples
print(train_df[
    ['utt', 'annot_utt', 'tokens', 'slot_tags']
    ].head()
)

                                   utt  \
1      wake me up at nine am on friday   
2  set an alarm for two hours from now   
4                           olly quiet   
5                                 stop   
6           olly pause for ten seconds   

                                           annot_utt  \
1  wake me up at [time : nine am] on [date : friday]   
2       set an alarm for [time : two hours from now]   
4                                         olly quiet   
5                                               stop   
6                olly pause for [time : ten seconds]   

                                         tokens  \
1      [wake, me, up, at, nine, am, on, friday]   
2  [set, an, alarm, for, two, hours, from, now]   
4                                 [olly, quiet]   
5                                        [stop]   
6              [olly, pause, for, ten, seconds]   

                                      slot_tags  
1       [O, O, O, O, B-time, I-time, O, B-date]  
2  

In [83]:
def normalize_text(text):
    return " ".join(text.lower().split())

# Check if there is any mismatch
def check_tokenization(split_df, split_name):
    mismatches = []
    
    for _, row in split_df.iterrows():
        reconstructed = " ".join(row["tokens"])
        
        if normalize_text(reconstructed) != normalize_text(row["utt"]):
            mismatches.append({"utt": row["utt"],
                              "reconstructed": reconstructed})
            
    print(f"{split_name} tokenization mismatched: "
          f"{len(mismatches)}")
    
    return mismatches

train_mismatches = check_tokenization(train_df, 'Train')
dev_mismtches = check_tokenization(dev_df, 'Dev')
test_mismatches = check_tokenization(test_df, 'Test')

train_mismatches[:10]

Train tokenization mismatched: 0
Dev tokenization mismatched: 0
Test tokenization mismatched: 0


[]

#### Task 1: Intent Classification using TF-IDF and Logistic Regression

In [84]:
# TF-IDF Features
tfidf = TfidfVectorizer(lowercase=True, ngram_range = (1,2))

# Fit vocab using TRAINING data only
X_train_utt = tfidf.fit_transform(train_df['utt'])

# Apply same vocab to dev and test
X_dev_utt = tfidf.transform(dev_df['utt'])
X_test_utt = tfidf.transform(test_df['utt'])

print("Train shape: ", X_train_utt.shape)
print("Dev shape: ", X_dev_utt.shape)
print("Test shape: ", X_test_utt.shape)

Train shape:  (11514, 29040)
Dev shape:  (2033, 29040)
Test shape:  (2974, 29040)


We represented each utterance using TF-IDF unigram and bigram features. 
The TF-IDF vocabulary was fitted using only the training split and applied to the validation and test splits.
Every utterance is represented using the same 29, 040 TF-IDF features.

In [85]:
# Logistic Regression

intent_model = LogisticRegression(max_iter=1000, random_state=42)

intent_model.fit(X_train_utt, train_labels)

dev_intent_predictions = intent_model.predict(X_dev_utt)

dev_intent_accuracy = accuracy_score(dev_labels, dev_intent_predictions) * 100

print(f"Validation Intent Accuracy: "
      f"{dev_intent_accuracy: .2f}%")

Validation Intent Accuracy:  79.69%


#### Task 2 Slot Filling using token features

In [86]:
# Create simple contextual features for each token
def token_features(tokens, i):
    features = {
        'word': tokens[i].lower()
    }
    
    # if current word is not the first word, so there is a previous word at i-1
    if i > 0:
        features['prev_word'] = tokens[i-1].lower()
    else:
    # i == 0 means beginning of sentence    
        features['BOS'] = True
    # if current word is not the last word, means there is a next word at i+1        
    if i < len(tokens) - 1:
        features['next_word'] = tokens[i+1].lower()
    else:
    # if last word means end of sentence
        features['EOS'] = True
    
    return features

# Build slot dataset: create one training example for each word
def build_slot_data(df):
    features = []
    labels = []
    
    for _, row in df.iterrows():
        tokens = row['tokens']
        tags = row['slot_tags']
        
        for i in range(len(tokens)):
            features.append(token_features(tokens, i))
            labels.append(tags[i])
            
    return features, labels

X_train_slot_features, y_train_slot = build_slot_data(train_df)
X_dev_slot_features, y_dev_slot = build_slot_data(dev_df)
X_test_slot_features, y_test_slot = build_slot_data(test_df)

print("Training tokens: ", len(X_train_slot_features))
print("Dev tokens: ", len(X_dev_slot_features))
print("Test tokens: ", len(X_test_slot_features))

Training tokens:  79732
Dev tokens:  13962
Test tokens:  20287


In [87]:
# Since Logistic Regression can't directly process words/features, need to convert to numbers

slot_vectorizer = DictVectorizer(sparse = True)

X_train_slot = slot_vectorizer.fit_transform(X_train_slot_features)
X_dev_slot = slot_vectorizer.transform(X_dev_slot_features)
X_test_slot = slot_vectorizer.transform(X_test_slot_features)

print("Train slot shape: ", X_train_slot.shape)
print("Dev slot shape: ", X_dev_slot.shape)
print("Test slot shape: ", X_test_slot.shape)

Train slot shape:  (79732, 14838)
Dev slot shape:  (13962, 14838)
Test slot shape:  (20287, 14838)


In [88]:
# Known slot types
all_slot_types = sorted({tag.split('-',1)[1]
                        for split_tags in[y_train_slot,y_dev_slot,y_test_slot]
                        for tag in split_tags
                        if tag != 'O'})

print("Number of slot types: ", len(all_slot_types))

#Create complete BIO label space
all_bio_labels = ['O']

for slot_type in all_slot_types:
    all_bio_labels.extend([f'B-{slot_type}', f'I-{slot_type}'])

# Convert BIO slot labels into numerical classes
slot_label_encoder = LabelEncoder()
slot_label_encoder.fit(all_bio_labels)
                           
# Encode labels
y_train_slot_encoded = slot_label_encoder.transform(y_train_slot)
y_dev_slot_encoded = slot_label_encoder.transform(y_dev_slot)
y_test_slot_encoded = slot_label_encoder.transform(y_test_slot)

print("Number of BIO slot labels: ", len(slot_label_encoder.classes_))
                           
missing_from_train = sorted(set(all_bio_labels) - set(y_train_slot))

print("Bio labels absent from training: ", len(missing_from_train))
print(missing_from_train)

Number of slot types:  55
Number of BIO slot labels:  111
Bio labels absent from training:  4
['I-cooking_type', 'I-email_folder', 'I-game_type', 'I-transport_type']


In [89]:
# Train the slot
slot_model = LogisticRegression(max_iter = 1000, random_state = 42)

slot_model.fit(X_train_slot, y_train_slot_encoded)

# Predict DEV only
dev_slot_predictions = slot_model.predict(X_dev_slot)

# Converting back to BIO labels
dev_predicted_tags = slot_label_encoder.inverse_transform(dev_slot_predictions)

print(dev_predicted_tags[:20])

['O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'O' 'B-house_place' 'O' 'O' 'O' 'O'
 'O' 'O' 'O' 'O' 'O']


Now, we have successfully show that there are 13, 1962 individual dev token predictions. We will need to change it back to 2, 033 utterances with its respective predicted BIO labels

In [90]:
def regroup_predictions(df, flat_predictions):
    grouped_predictions = []
    
    index = 0
    
    for tokens in df['tokens']:
        num_tokens = len(tokens)
        
        utterance_predictions = flat_predictions[index:index + num_tokens]
        
        grouped_predictions.append(list(utterance_predictions))
        
        index += num_tokens
        
    # Ensure every prediction was used
    assert index == len(flat_predictions)
        
    return grouped_predictions
    
dev_predicted_tags_grouped = regroup_predictions(dev_df, dev_predicted_tags)

print("Number of dev utterances: ", len(dev_predicted_tags_grouped))

assert len(dev_predicted_tags_grouped) == len(dev_df)

Number of dev utterances:  2033


In [91]:
# Check everything is aligned
assert all(len(tokens) == len(predictions)
          for tokens, predictions in zip(dev_df['tokens'], dev_predicted_tags_grouped))
print("All dev predictions align with their tokens.")

All dev predictions align with their tokens.


In [92]:
# Convert BIO tags back into slots
def bio_to_slots(tokens, tags):

    slots = []

    current_type = None

    current_words = []

    for token, tag in zip(tokens, tags):

        if tag == 'O':

            if current_type is not None:

                slots.append(

                    (current_type, ' '.join(current_words))

                )

                current_type = None

                current_words = []

        elif tag.startswith('B-'):

            # Finish previous slot, if any

            if current_type is not None:

                slots.append(

                    (current_type, ' '.join(current_words))

                )

            current_type = tag[2:]

            current_words = [token]

        elif tag.startswith('I-'):

            slot_type = tag[2:]

            # Continue the current slot if types match

            if current_type == slot_type:

                current_words.append(token)

            else:

                # Invalid I-tag:

                # treat it as beginning of a new slot

                if current_type is not None:

                    slots.append(

                        (current_type, ' '.join(current_words))

                    )

                current_type = slot_type

                current_words = [token]

    # Close final slot

    if current_type is not None:

        slots.append(

            (current_type, ' '.join(current_words))

        )

    return slots
    

In [93]:
# Just to test if the above function works
example_tokens = ['wake', 'me', 'up', 'at', 'five', 'am', 'this', 'week']

example_tags = ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'B-date', 'I-date']

print(bio_to_slots(example_tokens, example_tags))

[('time', 'five am'), ('date', 'this week')]


In [94]:
for i in range(len(dev_df)):
    gold_slots = bio_to_slots(dev_df.iloc[i]['tokens'], dev_df.iloc[i]['slot_tags'])
    
    if len(gold_slots) > 0:
        predicted_slots = bio_to_slots(dev_df.iloc[i]['tokens'], dev_predicted_tags_grouped[i])
        
        print("utt       : ", dev_df.iloc[i]['utt'])
        print("annot_utt: ", dev_df.iloc[i]['annot_utt'])
        print("tokens    : ", dev_df.iloc[i]['tokens'])
        print("tags      : ", dev_df.iloc[i]['slot_tags'])
        print("gold slots: ", gold_slots)
        print("\nPredicted slots: ", predicted_slots)
         
        if "six am" in dev_df.iloc[i]['utt']:
            break

utt       :  dim the lights in the hall
annot_utt:  dim the lights in the [house_place : hall]
tokens    :  ['dim', 'the', 'lights', 'in', 'the', 'hall']
tags      :  ['O', 'O', 'O', 'O', 'O', 'B-house_place']
gold slots:  [('house_place', 'hall')]

Predicted slots:  [('house_place', 'hall')]
utt       :  make a room darker
annot_utt:  make a [house_place : room] darker
tokens    :  ['make', 'a', 'room', 'darker']
tags      :  ['O', 'O', 'B-house_place', 'O']
gold slots:  [('house_place', 'room')]

Predicted slots:  []
utt       :  clean the flat
annot_utt:  clean the [house_place : flat]
tokens    :  ['clean', 'the', 'flat']
tags      :  ['O', 'O', 'B-house_place']
gold slots:  [('house_place', 'flat')]

Predicted slots:  []
utt       :  cleaning is good dust is so bad do now your magic clean my carpet
annot_utt:  cleaning is good dust is so bad do now your magic clean my [house_place : carpet]
tokens    :  ['cleaning', 'is', 'good', 'dust', 'is', 'so', 'bad', 'do', 'now', 'your', 'ma

#### Evaluating span

In [98]:
def calculate_slot_f1(df, predicted_tags_grouped):
    TP = 0
    FP = 0
    FN = 0
    
    for i in range(len(df)):
        tokens = df.iloc[i]['tokens']
        
        # Gold slots
        gold_slots = bio_to_slots(tokens, df.iloc[i]['slot_tags'])
        
        # Predicted slots
        predicted_slots = bio_to_slots(tokens, predicted_tags_grouped[i])
        
        # Counter handles duplicate slots safely
        gold_counter = Counter(gold_slots)
        predicted_counter = Counter(predicted_slots)
        
        # Exact matches only
        matches = gold_counter & predicted_counter
        
        tp = sum(matches.values())
        fp = sum(predicted_counter.values()) - tp
        fn = sum(gold_counter.values()) - tp
        
        TP += tp
        FP += fp
        FN += fn
        
    if(2 * TP + FP +FN) == 0:
        f1 = 0.0
    else:
        f1 = 100 * (2 * TP) / (2 * TP + FP + FN)
        
    return f1, TP, FP, FN

dev_slot_f1, TP, FP, FN = calculate_slot_f1(dev_df, dev_predicted_tags_grouped)

print(f"Validation Slot Micro F1: {dev_slot_f1: .2f}%")
print(f"True Positives: {TP}")
print(f"False Positives: {FP}")
print(f"False Negatives: {FN}")

dev_combined_score = (dev_intent_accuracy + dev_slot_f1) / 2
print(f"DEV Combined Validation Score: " f"{combined_score: .2f}%")

Validation Slot Micro F1:  66.67%
True Positives: 1229
False Positives: 446
False Negatives: 783
DEV Combined Validation Score:  73.18%


Baseline selected using the mean of Intent Accuracy and Slot F1

In [99]:
# Count Parameters

intent_params = (intent_model.coef_.size + intent_model.intercept_.size)

slot_params = (slot_model.coef_.size + slot_model.intercept_.size)

total_params = intent_params + slot_params

print(f"Intent model parameters: {intent_params:,}")
print(f"Slot model parameters: {slot_params:,}")
print(f"Total parameters: {total_params:,}")

assert total_params < 20_000_000

print("PASS: Model is within the 20M parameter limit.")

Intent model parameters: 1,742,460
Slot model parameters: 1,587,773
Total parameters: 3,330,233
PASS: Model is within the 20M parameter limit.


In [106]:
# Saving Checkpoint

import joblib

baseline_checkpoint = {
    'intent_model': intent_model,
    'slot_model': slot_model,
    'utterance_vectorizer': tfidf,
    'slot_vectorizer': slot_vectorizer,
    'intent_label_encoder': label_encoder,
    'slot_label_encoder': slot_label_encoder,
    'dev_intent_accuracy': dev_intent_accuracy,
    'dev_slot_f1': dev_slot_f1,
    'dev_combined_score': dev_combined_score
}

checkpoint_filename = 'stage1_baseline.pkl'

joblib.dump(baseline_checkpoint, checkpoint_filename)

print(f"Saved checkpoint: {checkpoint_filename}")

Saved checkpoint: stage1_baseline.pkl


In [107]:
loaded_checkpoint = joblib.load('stage1_baseline.pkl')
print("Stage 1 checkpoint loaded successfully.")


Stage 1 checkpoint loaded successfully.


In [11]:
# Run Intent Prediction for Time
start_time = time.time()
Y_pred = model.predict(X_test)
end_time = time.time()

total_predicted_time = end_time - start_time
avg_time_per_sample = (total_predicted_time / X_test.shape[0]) * 1000 # in milliseconds

# Run slot Prediction for F1
Y_pred_slot = slot_model.predict(X_test_slots)

# Calculate Final Matrics
intent_acc = accuracy_score(Y_test, Y_pred) * 100
slot_micro_f1 = f1_score(Y_test_slots, Y_pred_slot, average='micro') * 100

all_labels = range(len(label_encoder.classes_))

print("="*60)
print("TASK 1 & 2: Prediction Model using Logistic Regression results")
print("="*60)
# Following the Assignment format
print(f"{'Metric':<20} | {'Value':<20}")
print("-" * 42)
print(f"{'Intent accuracy':<20} | {intent_acc:.2f}%")
print(f"{'Slot micro F1':<20} | {slot_micro_f1:.2f}%")
print("-" * 42)

# Adding the time analysis below the required metrics
print(f"{'Total Pred Time':<20} | {total_predicted_time:.4f} s")
print(f"{'Avg Time per Utt':<20} | {avg_time_per_sample:.4f} ms")
print("="*60)

TASK 1 & 2: Prediction Model using Logistic Regression results
Metric               | Value               
------------------------------------------
Intent accuracy      | 78.08%
Slot micro F1        | 85.23%
------------------------------------------
Total Pred Time      | 0.0088 s
Avg Time per Utt     | 0.0029 ms
